# S6E10 Airline Passenger Satisfaction — Modelling & Training

**Metric:** ROC-AUC  
**Target:** `satisfaction` (1 = satisfied, 0 = not satisfied)  
**Feature set:** 23 columns from FE round 1 + `te_flight_distance` computed inside each fold (FE round 2)  
**Validation:** 5-fold StratifiedKFold (shuffle=True, random_state=42)  

---

## Contents
1. [Imports & Setup](#1-imports--setup)
2. [Load Data](#2-load-data)
3. [Target Encoding Helper](#3-target-encoding-helper)
4. [LightGBM Baseline](#4-lightgbm-baseline)
5. [CatBoost Baseline](#5-catboost-baseline)
6. [XGBoost Baseline](#6-xgboost-baseline)
7. [Baseline Ensemble & Correlation](#7-baseline-ensemble--correlation)
8. [LightGBM — Optuna Tuning](#8-lightgbm--optuna-tuning)
9. [LightGBM — Tuned (5-fold)](#9-lightgbm--tuned-5-fold)
10. [CatBoost — Optuna Tuning](#10-catboost--optuna-tuning)
11. [CatBoost — Tuned (5-fold)](#11-catboost--tuned-5-fold)
12. [Final Ensemble](#12-final-ensemble)
13. [Results Summary](#13-results-summary)
14. [Submissions](#14-submissions)

## 1. Imports & Setup

In [5]:
import os, numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import TargetEncoder
from catboost import CatBoostClassifier
import lightgbm as lgb
import xgboost as xgb
import optuna
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Works both locally and on Kaggle
DATA_DIR = Path('/kaggle/input/competitions/playground-series-s6e10') if Path('/kaggle/input/competitions/playground-series-s6e10').exists() \
           else Path('data/raw')

N_FOLDS = 5
RANDOM_STATE = 42
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

print(f"Data directory: {DATA_DIR}")

Data directory: /kaggle/input/competitions/playground-series-s6e10


## 2. Load Data & Preprocessing

Reads raw `train.csv` / `test.csv`, applies the same preprocessing as `features.ipynb` (missing-value flag, fillna, target encoding), then adds the two frequency features from FE round 1 (`flight_distance_freq`, `flight_distance_type_of_travel_freq`). The result is 23 features ready for modelling. The in-fold target encoding (`te_flight_distance`) is added inside each CV loop in sections 4–11.

In [6]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')

# Normalise column names: 'Inflight wifi service' -> 'inflight_wifi_service'
train.columns = train.columns.str.lower().str.replace(' ', '_')
test.columns  = test.columns.str.lower().str.replace(' ', '_')

# --- Preprocessing (identical to features.ipynb) ---

# 1. Missing-value flag — before fillna, while the gap information still exists
train['arrival_delay_missing'] = train['arrival_delay_in_minutes'].isna().astype(int)
test['arrival_delay_missing']  = test['arrival_delay_in_minutes'].isna().astype(int)

# 2. Fill gaps in arrival_delay with departure_delay (correlation 0.97)
train['arrival_delay_in_minutes'] = train['arrival_delay_in_minutes'].fillna(train['departure_delay_in_minutes'])
test['arrival_delay_in_minutes']  = test['arrival_delay_in_minutes'].fillna(test['departure_delay_in_minutes'])

# 3. Target: True/False -> 1/0
y_train = train['satisfaction'].astype(int)

# 4. Drop id, target, and arrival_delay (near-duplicate of departure_delay)
CAT_COLS  = ['gender', 'customer_type', 'type_of_travel', 'class']
DROP_COLS = ['id', 'satisfaction', 'arrival_delay_in_minutes']

X_raw      = train.drop(columns=DROP_COLS)
X_test_raw = test.drop(columns=['id', 'arrival_delay_in_minutes'])

# --- Add frequency features (E8 + E9 from features.ipynb round 1) ---

def add_features(train_df, test_df):
    """Add flight_distance_freq and flight_distance_type_of_travel_freq.
    Counts are computed over train+test together (no target involved)."""
    train_df = train_df.copy()
    test_df  = test_df.copy()
    full = pd.concat([train_df, test_df], ignore_index=True)
    n = len(train_df)

    # E8: frequency of the flight_distance value
    counts = full['flight_distance'].map(full['flight_distance'].value_counts())
    train_df['flight_distance_freq'] = counts.iloc[:n].values
    test_df['flight_distance_freq']  = counts.iloc[n:].values

    # E9: frequency of (flight_distance, type_of_travel) pair
    key = full['flight_distance'].astype(str) + '_' + full['type_of_travel'].astype(str)
    counts = key.map(key.value_counts())
    train_df['flight_distance_type_of_travel_freq'] = counts.iloc[:n].values
    test_df['flight_distance_type_of_travel_freq']  = counts.iloc[n:].values

    return train_df, test_df

X_train, X_test = add_features(X_raw, X_test_raw)

print(f'Train: {X_train.shape}, Test: {X_test.shape}, Target: {y_train.shape}')
print(f'Positive rate: {y_train.mean():.3f}')
print(f'Features: {list(X_train.columns)}')

Train: (699635, 23), Test: (299844, 23), Target: (699635,)
Positive rate: 0.444
Features: ['age', 'flight_distance', 'inflight_wifi_service', 'departure/arrival_time_convenient', 'ease_of_online_booking', 'gate_location', 'food_and_drink', 'online_boarding', 'seat_comfort', 'inflight_entertainment', 'on-board_service', 'leg_room_service', 'baggage_handling', 'checkin_service', 'cleanliness', 'departure_delay_in_minutes', 'gender', 'customer_type', 'type_of_travel', 'class', 'arrival_delay_missing', 'flight_distance_freq', 'flight_distance_type_of_travel_freq']


## 3. Target Encoding Helper

**Why inside the fold?**  
`te_flight_distance` encodes the target rate per distance value. If computed on the full train before splitting, the validation fold leaks its own target into the feature — CV score inflates and becomes unreliable. Computing it inside each fold (fit on train part only, transform val and test) avoids this leak.

**`cv=5` inside TargetEncoder:** the encoder itself uses cross-fitting on the training part so that even training rows get an out-of-fold estimate. This prevents the model from memorising the encoding.

In [8]:
def add_te_flight_distance(X_fit, y_fit, other_frames, smooth='auto'):
    """Add te_flight_distance column without target leakage.
    
    Call INSIDE a fold:
      X_fit / y_fit  — training part of the fold
      other_frames   — list of frames to transform with the fitted encoder
                       (validation part, test set)
    Returns: encoded X_fit, list of encoded other frames.
    """
    enc = TargetEncoder(smooth=smooth, cv=5, shuffle=True, random_state=RANDOM_STATE)
    key = lambda d: d[['flight_distance']].astype(str)

    X_fit = X_fit.copy()
    X_fit['te_flight_distance'] = enc.fit_transform(key(X_fit), y_fit)[:, 0]

    encoded = []
    for frame in other_frames:
        frame = frame.copy()
        frame['te_flight_distance'] = enc.transform(key(frame))[:, 0]
        encoded.append(frame)

    return X_fit, encoded

## 4. LightGBM Baseline

Baseline parameters from `features.ipynb`. Used as the reference point throughout FE (CV 0.95964 on 23 features without TE).

In [16]:
# Cast categoricals for LightGBM
X_train_lgbm = X_train.copy()
X_test_lgbm  = X_test.copy()
for col in CAT_COLS:
    X_train_lgbm[col] = X_train_lgbm[col].astype('category')
    X_test_lgbm[col]  = X_test_lgbm[col].astype('category')

LGBM_PARAMS = {
    'objective': 'binary', 'metric': 'auc',
    'learning_rate': 0.05, 'num_leaves': 127, 'min_child_samples': 20,
    'subsample': 0.8, 'colsample_bytree': 0.8,
    'n_estimators': 1000, 'early_stopping_rounds': 50,
    'verbose': -1, 'random_state': RANDOM_STATE,
    'device': 'gpu',
}

oof_lgbm  = np.zeros(len(X_train))
test_lgbm = np.zeros(len(X_test))
fold_scores_lgbm = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_lgbm, y_train), 1):
    X_tr, X_val = X_train_lgbm.iloc[tr_idx], X_train_lgbm.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    X_tr, (X_val, X_te) = add_te_flight_distance(X_tr, y_tr, [X_val, X_test_lgbm])

    model = lgb.LGBMClassifier(**LGBM_PARAMS)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], categorical_feature=CAT_COLS)

    val_pred = model.predict_proba(X_val)[:, 1]
    oof_lgbm[val_idx] = val_pred
    test_lgbm += model.predict_proba(X_te)[:, 1] / N_FOLDS

    score = roc_auc_score(y_val, val_pred)
    fold_scores_lgbm.append(score)
    print(f'Fold {fold} AUC: {score:.5f}')

cv_lgbm = roc_auc_score(y_train, oof_lgbm)
print(f'\nCV AUC (OOF): {cv_lgbm:.5f}')
print(f'Mean fold:    {np.mean(fold_scores_lgbm):.5f} ± {np.std(fold_scores_lgbm):.5f}')

1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


Fold 1 AUC: 0.96046
Fold 2 AUC: 0.95903
Fold 3 AUC: 0.96078
Fold 4 AUC: 0.96036
Fold 5 AUC: 0.96027

CV AUC (OOF): 0.96017
Mean fold:    0.96018 ± 0.00060


## 5. CatBoost Baseline

CatBoost uses **oblivious (symmetric) trees**: all branches at the same depth make the same split. This differs from LightGBM's leaf-wise growth. `depth=6` (64 leaves) is the standard working range for CatBoost — comparable in regularisation strength to `num_leaves=127` in LightGBM.

CatBoost accepts string categoricals natively and builds its own internal target statistics (CTR/Counter) per category and combination. No manual casting to `category` dtype needed.

In [17]:
CAT_PARAMS = {
    'iterations': 1000, 'learning_rate': 0.05, 'depth': 6,
    'loss_function': 'Logloss', 'eval_metric': 'AUC',
    'early_stopping_rounds': 50, 'random_seed': RANDOM_STATE,
    'thread_count': -1, 'verbose': 0,
    'task_type': 'GPU',
}

oof_cat  = np.zeros(len(X_train))
test_cat = np.zeros(len(X_test))
fold_scores_cat = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    X_tr, (X_val, X_te) = add_te_flight_distance(X_tr, y_tr, [X_val, X_test])

    model = CatBoostClassifier(**CAT_PARAMS, cat_features=CAT_COLS)
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val))

    val_pred = model.predict_proba(X_val)[:, 1]
    oof_cat[val_idx] = val_pred
    test_cat += model.predict_proba(X_te)[:, 1] / N_FOLDS

    score = roc_auc_score(y_val, val_pred)
    fold_scores_cat.append(score)
    print(f'Fold {fold} AUC: {score:.5f}')

cv_cat = roc_auc_score(y_train, oof_cat)
print(f'\nCV AUC (OOF): {cv_cat:.5f}')
print(f'Mean fold:    {np.mean(fold_scores_cat):.5f} ± {np.std(fold_scores_cat):.5f}')

Default metric period is 5 because AUC is/are not implemented for GPU


Fold 1 AUC: 0.95965


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 2 AUC: 0.95832


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 3 AUC: 0.96033


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 4 AUC: 0.95976


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 5 AUC: 0.95967

CV AUC (OOF): 0.95954
Mean fold:    0.95954 ± 0.00066


## 6. XGBoost Baseline

`tree_method='hist'` enables histogram-based splitting — the same algorithm LightGBM uses by default. Without it, XGBoost would be ~10× slower on this dataset size.

`enable_categorical=True` requires columns to have pandas `category` dtype.

In [18]:
# Cast categoricals for XGBoost
X_train_xgb = X_train.copy()
X_test_xgb  = X_test.copy()
for col in CAT_COLS:
    X_train_xgb[col] = X_train_xgb[col].astype('category')
    X_test_xgb[col]  = X_test_xgb[col].astype('category')

XGB_PARAMS = {
    'n_estimators': 1000, 'learning_rate': 0.05, 'max_depth': 6,
    'subsample': 0.8, 'colsample_bytree': 0.8,
    'tree_method': 'hist', 'enable_categorical': True,
    'device': 'cuda',
    'eval_metric': 'auc', 'early_stopping_rounds': 50,
    'random_state': RANDOM_STATE,
}

oof_xgb  = np.zeros(len(X_train))
test_xgb = np.zeros(len(X_test))
fold_scores_xgb = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_xgb, y_train), 1):
    X_tr, X_val = X_train_xgb.iloc[tr_idx], X_train_xgb.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    X_tr, (X_val, X_te) = add_te_flight_distance(X_tr, y_tr, [X_val, X_test_xgb])

    model = xgb.XGBClassifier(**XGB_PARAMS)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=0)

    val_pred = model.predict_proba(X_val)[:, 1]
    oof_xgb[val_idx] = val_pred
    test_xgb += model.predict_proba(X_te)[:, 1] / N_FOLDS

    score = roc_auc_score(y_val, val_pred)
    fold_scores_xgb.append(score)
    print(f'Fold {fold} AUC: {score:.5f}')

cv_xgb = roc_auc_score(y_train, oof_xgb)
print(f'\nCV AUC (OOF): {cv_xgb:.5f}')
print(f'Mean fold:    {np.mean(fold_scores_xgb):.5f} ± {np.std(fold_scores_xgb):.5f}')

Fold 1 AUC: 0.96032
Fold 2 AUC: 0.95878
Fold 3 AUC: 0.96059
Fold 4 AUC: 0.96008
Fold 5 AUC: 0.95999

CV AUC (OOF): 0.95995
Mean fold:    0.95995 ± 0.00062


## 7. Baseline Ensemble & Correlation

Before tuning, we check whether the three models are diverse enough to benefit from ensembling.

**OOF correlation** tells us how similarly models rank the same rows. Correlation > 0.99 means models make nearly identical errors — averaging them gives little gain over the best single model.

In [19]:
oof_df = pd.DataFrame({'lgbm': oof_lgbm, 'xgb': oof_xgb, 'cat': oof_cat})
print('OOF prediction correlation:')
print(oof_df.corr().round(4))

oof_ensemble_base  = (oof_lgbm + oof_xgb + oof_cat) / 3
test_ensemble_base = (test_lgbm + test_xgb + test_cat) / 3

cv_ensemble_base = roc_auc_score(y_train, oof_ensemble_base)
print(f'\nBaseline ensemble CV AUC: {cv_ensemble_base:.5f}')

print(f'\n--- Baseline summary ---')
print(f'LightGBM : {cv_lgbm:.5f}')
print(f'XGBoost  : {cv_xgb:.5f}')
print(f'CatBoost : {cv_cat:.5f}')
print(f'Ensemble : {cv_ensemble_base:.5f}')

OOF prediction correlation:
        lgbm     xgb     cat
lgbm  1.0000  0.9980  0.9973
xgb   0.9980  1.0000  0.9983
cat   0.9973  0.9983  1.0000

Baseline ensemble CV AUC: 0.96020

--- Baseline summary ---
LightGBM : 0.96017
XGBoost  : 0.95995
CatBoost : 0.95954
Ensemble : 0.96020


**Conclusion:** OOF correlations are 0.997–0.998 — all three models find the same patterns. Simple averaging does not improve over the best single model. Hyperparameter tuning is the next lever.

## 8. LightGBM — Optuna Tuning

**Search space:**
- `learning_rate` — log scale (equal probability for 0.01–0.02 and 0.05–0.1)
- `num_leaves` — controls tree complexity (LightGBM grows leaf-wise)
- `reg_alpha / reg_lambda` — L1 and L2 regularisation, not used in the baseline

**3-fold CV** is used inside Optuna (instead of 5-fold) to keep runtime at ~90 min for 50 trials.
Final evaluation of the best parameters is done on 5-fold in section 9.

In [20]:
def objective_lgbm(trial):
    params = {
        'objective': 'binary', 'metric': 'auc',
        'verbose': -1, 'random_state': RANDOM_STATE,
        'n_estimators': 1000, 'early_stopping_rounds': 50,
        'device': 'gpu',
        'learning_rate':      trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves':         trial.suggest_int('num_leaves', 63, 255),
        'min_child_samples':  trial.suggest_int('min_child_samples', 10, 100),
        'subsample':          trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':   trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha':          trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':         trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
    }
    skf_opt = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    scores = []
    for tr_idx, val_idx in skf_opt.split(X_train_lgbm, y_train):
        X_tr, X_val = X_train_lgbm.iloc[tr_idx], X_train_lgbm.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        X_tr, (X_val, _) = add_te_flight_distance(X_tr, y_tr, [X_val, X_test_lgbm])
        model = lgb.LGBMClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], categorical_feature=CAT_COLS)
        scores.append(roc_auc_score(y_val, model.predict_proba(X_val)[:, 1]))
    return np.mean(scores)

study_lgbm = optuna.create_study(direction='maximize')
study_lgbm.optimize(objective_lgbm, n_trials=50, show_progress_bar=True)

print(f'Best CV (3-fold): {study_lgbm.best_value:.5f}')
print(f'Best params: {study_lgbm.best_params}')

  0%|          | 0/50 [00:00<?, ?it/s]

Best CV (3-fold): 0.96047
Best params: {'learning_rate': 0.011594054946533571, 'num_leaves': 232, 'min_child_samples': 69, 'subsample': 0.9851580134791578, 'colsample_bytree': 0.6505963546420028, 'reg_alpha': 1.5621237403785044, 'reg_lambda': 1.7672411530478345}


## 9. LightGBM — Tuned (5-fold)

Re-train with the best parameters found by Optuna on the full 5-fold scheme for a fair comparison with baselines.

In [21]:
LGBM_OPT_PARAMS = {
    'objective': 'binary', 'metric': 'auc',
    'verbose': -1, 'random_state': RANDOM_STATE,
    'n_estimators': 1000, 'early_stopping_rounds': 50,
    **study_lgbm.best_params,
}

oof_lgbm_opt  = np.zeros(len(X_train))
test_lgbm_opt = np.zeros(len(X_test))
fold_scores_lgbm_opt = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_lgbm, y_train), 1):
    X_tr, X_val = X_train_lgbm.iloc[tr_idx], X_train_lgbm.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    X_tr, (X_val, X_te) = add_te_flight_distance(X_tr, y_tr, [X_val, X_test_lgbm])

    model = lgb.LGBMClassifier(**LGBM_OPT_PARAMS)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], categorical_feature=CAT_COLS)

    val_pred = model.predict_proba(X_val)[:, 1]
    oof_lgbm_opt[val_idx] = val_pred
    test_lgbm_opt += model.predict_proba(X_te)[:, 1] / N_FOLDS

    score = roc_auc_score(y_val, val_pred)
    fold_scores_lgbm_opt.append(score)
    print(f'Fold {fold} AUC: {score:.5f}')

cv_lgbm_opt = roc_auc_score(y_train, oof_lgbm_opt)
print(f'\nCV AUC (OOF): {cv_lgbm_opt:.5f}')
print(f'Mean fold:    {np.mean(fold_scores_lgbm_opt):.5f} ± {np.std(fold_scores_lgbm_opt):.5f}')
print(f'vs baseline:  {cv_lgbm_opt - cv_lgbm:+.5f}')

Fold 1 AUC: 0.96098
Fold 2 AUC: 0.95936
Fold 3 AUC: 0.96123
Fold 4 AUC: 0.96066
Fold 5 AUC: 0.96058

CV AUC (OOF): 0.96056
Mean fold:    0.96057 ± 0.00065
vs baseline:  +0.00038


## 10. CatBoost — Optuna Tuning

CatBoost-specific parameters:
- `depth` — tree depth (capped at 6 for speed; oblivious trees grow exponentially slower with depth)
- `l2_leaf_reg` — L2 regularisation on leaf weights
- `bagging_temperature` — controls randomness of Bayesian bootstrap (0 = no randomness)
- `random_strength` — adds noise to split scores to reduce overfitting

**30% subsample** of train used for speed (full dataset would take ~25 hours for 50 trials).

In [22]:
X_opt, _, y_opt, _ = train_test_split(
    X_train, y_train, train_size=0.3, stratify=y_train, random_state=RANDOM_STATE
)

def objective_cat(trial):
    params = {
        'iterations': 1000, 'loss_function': 'Logloss', 'eval_metric': 'AUC',
        'early_stopping_rounds': 50, 'random_seed': RANDOM_STATE,
        'verbose': 0, 'thread_count': -1,
        'task_type': 'GPU',
        'learning_rate':      trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth':              trial.suggest_int('depth', 4, 6),
        'l2_leaf_reg':        trial.suggest_float('l2_leaf_reg', 1e-2, 10.0, log=True),
        'bagging_temperature':trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_strength':    trial.suggest_float('random_strength', 0.0, 1.0),
        'min_data_in_leaf':   trial.suggest_int('min_data_in_leaf', 1, 50),
    }
    skf_opt = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    scores = []
    for tr_idx, val_idx in skf_opt.split(X_opt, y_opt):
        X_tr, X_val = X_opt.iloc[tr_idx], X_opt.iloc[val_idx]
        y_tr, y_val = y_opt.iloc[tr_idx], y_opt.iloc[val_idx]
        X_tr, (X_val, _) = add_te_flight_distance(X_tr, y_tr, [X_val, X_test])
        model = CatBoostClassifier(**params, cat_features=CAT_COLS)
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val))
        scores.append(roc_auc_score(y_val, model.predict_proba(X_val)[:, 1]))
    return np.mean(scores)

study_cat = optuna.create_study(direction='maximize')
study_cat.optimize(objective_cat, n_trials=50, show_progress_bar=True)

print(f'Best CV (3-fold, 30% data): {study_cat.best_value:.5f}')
print(f'Best params: {study_cat.best_params}')

  0%|          | 0/50 [00:00<?, ?it/s]

Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric perio

Best CV (3-fold, 30% data): 0.95875
Best params: {'learning_rate': 0.04336429500029207, 'depth': 6, 'l2_leaf_reg': 0.024485951255744317, 'bagging_temperature': 0.43557390422246217, 'random_strength': 0.8336105913113857, 'min_data_in_leaf': 13}


## 11. CatBoost — Tuned (5-fold)

In [23]:
CAT_OPT_PARAMS = {
    'iterations': 1000, 'loss_function': 'Logloss', 'eval_metric': 'AUC',
    'early_stopping_rounds': 50, 'random_seed': RANDOM_STATE,
    'thread_count': -1, 'verbose': 0,
    **study_cat.best_params,
}

oof_cat_opt  = np.zeros(len(X_train))
test_cat_opt = np.zeros(len(X_test))
fold_scores_cat_opt = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    X_tr, (X_val, X_te) = add_te_flight_distance(X_tr, y_tr, [X_val, X_test])

    model = CatBoostClassifier(**CAT_OPT_PARAMS, cat_features=CAT_COLS)
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val))

    val_pred = model.predict_proba(X_val)[:, 1]
    oof_cat_opt[val_idx] = val_pred
    test_cat_opt += model.predict_proba(X_te)[:, 1] / N_FOLDS

    score = roc_auc_score(y_val, val_pred)
    fold_scores_cat_opt.append(score)
    print(f'Fold {fold} AUC: {score:.5f}')

cv_cat_opt = roc_auc_score(y_train, oof_cat_opt)
print(f'\nCV AUC (OOF): {cv_cat_opt:.5f}')
print(f'Mean fold:    {np.mean(fold_scores_cat_opt):.5f} ± {np.std(fold_scores_cat_opt):.5f}')
print(f'vs baseline:  {cv_cat_opt - cv_cat:+.5f}')

Fold 1 AUC: 0.95975
Fold 2 AUC: 0.95837
Fold 3 AUC: 0.96040
Fold 4 AUC: 0.95988
Fold 5 AUC: 0.95980

CV AUC (OOF): 0.95963
Mean fold:    0.95964 ± 0.00068
vs baseline:  +0.00010


## 12. Final Ensemble

We check OOF correlation between the two tuned models and test all ensemble combinations.

In [24]:
oof_df_final = pd.DataFrame({
    'lgbm_opt': oof_lgbm_opt,
    'cat_opt':  oof_cat_opt,
    'xgb':      oof_xgb,
})
print('OOF correlation (tuned models):')
print(oof_df_final.corr().round(4))

# Simple averages
combos = {
    'lgbm_opt + cat_opt':       (oof_lgbm_opt + oof_cat_opt) / 2,
    'lgbm_opt + xgb':           (oof_lgbm_opt + oof_xgb) / 2,
    'lgbm_opt + cat_opt + xgb': (oof_lgbm_opt + oof_cat_opt + oof_xgb) / 3,
}
print('\nEnsemble CV AUC:')
for name, oof in combos.items():
    print(f'  {name}: {roc_auc_score(y_train, oof):.5f}')

# Best ensemble for submission
oof_final  = (oof_lgbm_opt + oof_cat_opt) / 2
test_final = (test_lgbm_opt + test_cat_opt) / 2
cv_final = roc_auc_score(y_train, oof_final)
print(f'\nSelected ensemble CV AUC: {cv_final:.5f}')

OOF correlation (tuned models):
          lgbm_opt  cat_opt     xgb
lgbm_opt    1.0000   0.9974  0.9983
cat_opt     0.9974   1.0000  0.9983
xgb         0.9983   0.9983  1.0000

Ensemble CV AUC:
  lgbm_opt + cat_opt: 0.96035
  lgbm_opt + xgb: 0.96042
  lgbm_opt + cat_opt + xgb: 0.96034

Selected ensemble CV AUC: 0.96035


## 13. Results Summary

In [25]:
results = pd.DataFrame([
    {'Model': 'LightGBM baseline',    'CV AUC': cv_lgbm},
    {'Model': 'CatBoost baseline',    'CV AUC': cv_cat},
    {'Model': 'XGBoost baseline',     'CV AUC': cv_xgb},
    {'Model': 'Baseline ensemble',    'CV AUC': cv_ensemble_base},
    {'Model': 'LightGBM + Optuna',    'CV AUC': cv_lgbm_opt},
    {'Model': 'CatBoost + Optuna',    'CV AUC': cv_cat_opt},
    {'Model': 'Final ensemble',       'CV AUC': cv_final},
])
results['Delta vs LGBM baseline'] = results['CV AUC'] - cv_lgbm
results = results.sort_values('CV AUC', ascending=False).reset_index(drop=True)
print(results.to_string(index=False, float_format='{:.5f}'.format))

            Model  CV AUC  Delta vs LGBM baseline
LightGBM + Optuna 0.96056                 0.00038
   Final ensemble 0.96035                 0.00018
Baseline ensemble 0.96020                 0.00003
LightGBM baseline 0.96017                 0.00000
 XGBoost baseline 0.95995                -0.00023
CatBoost + Optuna 0.95963                -0.00054
CatBoost baseline 0.95954                -0.00064


## 14. Submissions

In [26]:
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')

# LightGBM tuned
sub_lgbm = sample_sub.copy()
sub_lgbm['satisfaction'] = test_lgbm_opt
sub_lgbm.to_csv('submission_lgbm_optuna_te.csv', index=False)
print(f'submission_lgbm_optuna_te.csv — CV {cv_lgbm_opt:.5f}')

# CatBoost tuned
sub_cat = sample_sub.copy()
sub_cat['satisfaction'] = test_cat_opt
sub_cat.to_csv('submission_cat_optuna_te.csv', index=False)
print(f'submission_cat_optuna_te.csv  — CV {cv_cat_opt:.5f}')

# Final ensemble
sub_final = sample_sub.copy()
sub_final['satisfaction'] = test_final
sub_final.to_csv('submission_ensemble_final.csv', index=False)
print(f'submission_ensemble_final.csv — CV {cv_final:.5f}')

submission_lgbm_optuna_te.csv — CV 0.96056
submission_cat_optuna_te.csv  — CV 0.95963
submission_ensemble_final.csv — CV 0.96035


In [9]:
for col in CAT_COLS:
    X_train[col] = X_train[col].astype('category')
    X_test[col]  = X_test[col].astype('category')

LGBM_OPT_PARAMS = {
    'objective': 'binary', 'metric': 'auc',
    'verbose': -1, 'random_state': RANDOM_STATE,
    'n_estimators': 1000, 'early_stopping_rounds': 50,
    'learning_rate': 0.011594054946533571, 'num_leaves': 232,
    'min_child_samples': 69, 'subsample': 0.9851580134791578,
    'colsample_bytree': 0.6505963546420028,
    'reg_alpha': 1.5621237403785044, 'reg_lambda': 1.7672411530478345,
}

oof_lgbm_opt  = np.zeros(len(X_train))
test_lgbm_opt = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    X_tr, (X_val, X_te) = add_te_flight_distance(X_tr, y_tr, [X_val, X_test])

    model = lgb.LGBMClassifier(**LGBM_OPT_PARAMS)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], categorical_feature=CAT_COLS)

    oof_lgbm_opt[val_idx] = model.predict_proba(X_val)[:, 1]
    test_lgbm_opt += model.predict_proba(X_te)[:, 1] / N_FOLDS
    print(f'Fold {fold} AUC: {roc_auc_score(y_val, oof_lgbm_opt[val_idx]):.5f}')

print(f'CV AUC (OOF): {roc_auc_score(y_train, oof_lgbm_opt):.5f}')

sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
sub['satisfaction'] = test_lgbm_opt
sub.to_csv('/kaggle/working/submission_lgbm_optuna_te.csv', index=False)
np.save('/kaggle/working/oof_lgbm_opt.npy', oof_lgbm_opt)
np.save('/kaggle/working/test_lgbm_opt.npy', test_lgbm_opt)
print(os.listdir('/kaggle/working'))

Fold 1 AUC: 0.96098
Fold 2 AUC: 0.95936
Fold 3 AUC: 0.96123
Fold 4 AUC: 0.96066
Fold 5 AUC: 0.96058
CV AUC (OOF): 0.96056
['oof_lgbm_opt.npy', 'test_lgbm_opt.npy', 'submission_lgbm_optuna_te.csv', '.virtual_documents']


In [13]:
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/competitions/playground-series-s6e10/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e10/train.csv
/kaggle/input/competitions/playground-series-s6e10/test.csv
/kaggle/input/datasets/deonissx/s6e10-orig-data/orig_data.csv


In [14]:
ORIG_PATH = '/kaggle/input/datasets/deonissx/s6e10-orig-data/orig_data.csv'
orig = pd.read_csv(ORIG_PATH)
orig.columns = orig.columns.str.lower().str.replace(' ', '_')

feats = [c for c in train.columns if c in orig.columns and c not in ('id', 'satisfaction', 'arrival_delay_missing')]
print(orig.shape, '| positive rate:', round(orig['satisfaction'].astype(int).mean(), 3))
print('features used by the orig model:', len(feats))
print('missing in orig:', [c for c in train.columns if c not in orig.columns])

(129880, 22) | positive rate: 0.434
features used by the orig model: 21
missing in orig: ['id', 'arrival_delay_missing']


In [15]:
# --- 1. Original-only model (parameters from the reference notebook) ---
orig_f = orig.copy()
orig_f['arrival_delay_in_minutes'] = orig_f['arrival_delay_in_minutes'].fillna(orig_f['departure_delay_in_minutes'])
orig_y = orig_f['satisfaction'].astype(int)

def to_cat(df, col, cats):
    return pd.Categorical(df[col], categories=cats)

orig_X  = orig_f[feats].copy()
tr_orig = train[feats].copy()
te_orig = test[feats].copy()
for col in CAT_COLS:
    cats = sorted(set(orig_X[col].astype(str)) | set(tr_orig[col].astype(str)) | set(te_orig[col].astype(str)))
    for df in (orig_X, tr_orig, te_orig):
        df[col] = pd.Categorical(df[col].astype(str), categories=cats)

m_orig = lgb.LGBMClassifier(
    objective='binary', n_estimators=600, learning_rate=0.05, max_depth=8,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.5,
    reg_lambda=5.0, reg_alpha=0.05, random_state=123, n_jobs=-1, verbosity=-1)
m_orig.fit(orig_X, orig_y, categorical_feature=CAT_COLS)

p_tr = m_orig.predict_proba(tr_orig)[:, 1]
p_te = m_orig.predict_proba(te_orig)[:, 1]
print('AUC of the orig-model on competition train:', round(roc_auc_score(y_train, p_tr), 5))

logit = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / np.clip(1 - p, 1e-6, 1 - 1e-6))
X_o, X_test_o = X_train.copy(), X_test.copy()
X_o['orig_prob'], X_o['orig_logit'] = p_tr, logit(p_tr)
X_test_o['orig_prob'], X_test_o['orig_logit'] = p_te, logit(p_te)

# --- 2. LightGBM baseline with in-fold TE (same params as section 4, CPU) ---
for col in CAT_COLS:
    X_o[col] = X_o[col].astype('category')
    X_test_o[col] = X_test_o[col].astype('category')

P = {'objective': 'binary', 'metric': 'auc', 'learning_rate': 0.05, 'num_leaves': 127,
     'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.8,
     'n_estimators': 1000, 'early_stopping_rounds': 50, 'verbose': -1, 'random_state': RANDOM_STATE}

oof_o = np.zeros(len(X_o))
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_o, y_train), 1):
    X_tr, X_val = X_o.iloc[tr_idx], X_o.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    X_tr, (X_val,) = add_te_flight_distance(X_tr, y_tr, [X_val])
    model = lgb.LGBMClassifier(**P)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], categorical_feature=CAT_COLS)
    oof_o[val_idx] = model.predict_proba(X_val)[:, 1]
    print(f'Fold {fold} AUC: {roc_auc_score(y_val, oof_o[val_idx]):.5f}')

cv_o = roc_auc_score(y_train, oof_o)
print(f'\nCV AUC (OOF): {cv_o:.5f} | vs LGBM baseline with TE (0.96017): {cv_o - 0.96017:+.5f}')

AUC of the orig-model on competition train: 0.95472
Fold 1 AUC: 0.96045
Fold 2 AUC: 0.95911
Fold 3 AUC: 0.96114
Fold 4 AUC: 0.96038
Fold 5 AUC: 0.96037

CV AUC (OOF): 0.96027 | vs LGBM baseline with TE (0.96017): +0.00010


In [16]:
SEEDS = [42, 7, 123, 2024, 777]
W = Path('/kaggle/working')

oofs, tests = [], []
for seed in SEEDS:
    f_oof, f_test = W / f'oof_lgbm_seed{seed}.npy', W / f'test_lgbm_seed{seed}.npy'

    if f_oof.exists() and f_test.exists():          # resume after a restart
        oof_s, test_s = np.load(f_oof), np.load(f_test)
    else:
        params = {**LGBM_OPT_PARAMS, 'random_state': seed}
        oof_s, test_s = np.zeros(len(X_train)), np.zeros(len(X_test))
        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
            X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
            y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
            X_tr, (X_val, X_te) = add_te_flight_distance(X_tr, y_tr, [X_val, X_test])

            model = lgb.LGBMClassifier(**params)
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], categorical_feature=CAT_COLS)
            oof_s[val_idx] = model.predict_proba(X_val)[:, 1]
            test_s += model.predict_proba(X_te)[:, 1] / N_FOLDS
        np.save(f_oof, oof_s)
        np.save(f_test, test_s)

    oofs.append(oof_s)
    tests.append(test_s)
    print(f'seed {seed}: CV {roc_auc_score(y_train, oof_s):.5f} | '
          f'average of {len(oofs)} seeds: {roc_auc_score(y_train, np.mean(oofs, axis=0)):.5f}')

oof_avg, test_avg = np.mean(oofs, axis=0), np.mean(tests, axis=0)
print(f'\nFinal CV ({len(SEEDS)} seeds): {roc_auc_score(y_train, oof_avg):.5f}  (single seed 42: 0.96056)')

sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
sub['satisfaction'] = test_avg
sub.to_csv(W / 'submission_lgbm_5seeds.csv', index=False)
np.save(W / 'oof_lgbm_avg.npy', oof_avg)
np.save(W / 'test_lgbm_avg.npy', test_avg)
print(os.listdir(W))

seed 42: CV 0.96056 | average of 1 seeds: 0.96056
seed 7: CV 0.96053 | average of 2 seeds: 0.96057
seed 123: CV 0.96054 | average of 3 seeds: 0.96058
seed 2024: CV 0.96056 | average of 4 seeds: 0.96059
seed 777: CV 0.96052 | average of 5 seeds: 0.96058

Final CV (5 seeds): 0.96058  (single seed 42: 0.96056)
['test_lgbm_seed777.npy', 'test_lgbm_seed42.npy', 'test_lgbm_seed2024.npy', 'oof_lgbm_seed7.npy', 'test_lgbm_seed123.npy', 'oof_lgbm_avg.npy', 'oof_lgbm_opt.npy', 'test_lgbm_opt.npy', 'oof_lgbm_seed42.npy', 'test_lgbm_avg.npy', 'submission_lgbm_5seeds.csv', 'oof_lgbm_seed2024.npy', 'submission_lgbm_optuna_te.csv', 'test_lgbm_seed7.npy', 'oof_lgbm_seed777.npy', 'oof_lgbm_seed123.npy', '.virtual_documents']


In [17]:
# --- route profile: per-distance mean of every other feature (no target; train+test together) ---
full = pd.concat([X_train, X_test], ignore_index=True)
n = len(X_train)
prof_cols = [c for c in X_train.columns
             if c not in ('flight_distance', 'flight_distance_freq', 'flight_distance_type_of_travel_freq')]
coded = full[prof_cols].copy()
for c in CAT_COLS:
    coded[c] = coded[c].astype('category').cat.codes
coded['flight_distance'] = full['flight_distance'].values
grp = coded.groupby('flight_distance')
PROF = pd.DataFrame({f'fdm_{c}': grp[c].transform('mean').astype('float32') for c in prof_cols})
print('profile columns:', PROF.shape[1])

X_prof = X_train.copy()
for c in PROF.columns:
    X_prof[c] = PROF[c].values[:n]

X_prof_orig = X_prof.copy()
X_prof_orig['orig_prob'], X_prof_orig['orig_logit'] = p_tr, logit(p_tr)

BASE_FOLDS = [0.96046, 0.95903, 0.96078, 0.96036, 0.96027]      # LGBM baseline + TE (section 4)

def cv_lgbm(X, label):
    X = X.copy()
    for col in CAT_COLS:
        X[col] = X[col].astype('category')
    oof, scores = np.zeros(len(X)), []
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y_train), 1):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        X_tr, (X_val,) = add_te_flight_distance(X_tr, y_tr, [X_val])
        model = lgb.LGBMClassifier(**P)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], categorical_feature=CAT_COLS)
        oof[val_idx] = model.predict_proba(X_val)[:, 1]
        scores.append(roc_auc_score(y_val, oof[val_idx]))
    cv = roc_auc_score(y_train, oof)
    print(f'{label}: CV {cv:.5f} | vs baseline 0.96017: {cv - 0.96017:+.5f} | '
          f'vs orig-only 0.96027: {cv - 0.96027:+.5f}')
    print('   fold deltas vs baseline:', ' '.join(f'{s - b:+.5f}' for s, b in zip(scores, BASE_FOLDS)))
    return oof

oof_prof      = cv_lgbm(X_prof,      'route profile only')
oof_prof_orig = cv_lgbm(X_prof_orig, 'route profile + orig model')

profile columns: 20
route profile only: CV 0.96036 | vs baseline 0.96017: +0.00019 | vs orig-only 0.96027: +0.00009
   fold deltas vs baseline: +0.00017 +0.00030 +0.00028 +0.00005 +0.00013
route profile + orig model: CV 0.96048 | vs baseline 0.96017: +0.00031 | vs orig-only 0.96027: +0.00021
   fold deltas vs baseline: +0.00019 +0.00042 +0.00034 +0.00029 +0.00036


In [18]:
X_final = X_prof_orig.copy()
X_test_final = X_test.copy()
for c in PROF.columns:
    X_test_final[c] = PROF[c].values[n:]
X_test_final['orig_prob'], X_test_final['orig_logit'] = p_te, logit(p_te)
for col in CAT_COLS:
    X_final[col] = X_final[col].astype('category')
    X_test_final[col] = X_test_final[col].astype('category')

TUNED_FOLDS = [0.96098, 0.95936, 0.96123, 0.96066, 0.96058]     # tuned LGBM without new features
oof_new, test_new, scores = np.zeros(len(X_final)), np.zeros(len(X_test_final)), []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_final, y_train), 1):
    X_tr, X_val = X_final.iloc[tr_idx], X_final.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    X_tr, (X_val, X_te) = add_te_flight_distance(X_tr, y_tr, [X_val, X_test_final])

    model = lgb.LGBMClassifier(**LGBM_OPT_PARAMS)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], categorical_feature=CAT_COLS)
    oof_new[val_idx] = model.predict_proba(X_val)[:, 1]
    test_new += model.predict_proba(X_te)[:, 1] / N_FOLDS
    scores.append(roc_auc_score(y_val, oof_new[val_idx]))
    print(f'Fold {fold} AUC: {scores[-1]:.5f}')

cv_new = roc_auc_score(y_train, oof_new)
print(f'\nCV AUC (OOF): {cv_new:.5f} | vs tuned LGBM 0.96056: {cv_new - 0.96056:+.5f}')
print('fold deltas:', ' '.join(f'{s - b:+.5f}' for s, b in zip(scores, TUNED_FOLDS)))

sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
sub['satisfaction'] = test_new
sub.to_csv('/kaggle/working/submission_lgbm_prof_orig.csv', index=False)
np.save('/kaggle/working/oof_lgbm_prof_orig.npy', oof_new)
np.save('/kaggle/working/test_lgbm_prof_orig.npy', test_new)

Fold 1 AUC: 0.96095
Fold 2 AUC: 0.95984
Fold 3 AUC: 0.96153
Fold 4 AUC: 0.96080
Fold 5 AUC: 0.96057

CV AUC (OOF): 0.96072 | vs tuned LGBM 0.96056: +0.00016
fold deltas: -0.00003 +0.00048 +0.00030 +0.00014 -0.00001


In [19]:
import numpy as np, pandas as pd, lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

BASE = {'objective':'binary','metric':'auc','verbose':-1,'random_state':RANDOM_STATE,
        'n_estimators':1000,'early_stopping_rounds':50,'learning_rate':0.05,
        'num_leaves':127,'min_child_samples':20,'colsample_bytree':0.8}

def run_cv(X, params=BASE):
    oof = np.zeros(len(X)); fold_auc = []
    for tr, va in skf.split(X, y_train):
        Xtr, [Xva] = add_te_flight_distance(X.iloc[tr], y_train.iloc[tr], [X.iloc[va]])
        m = LGBMClassifier(**params)
        m.fit(Xtr, y_train.iloc[tr], eval_set=[(Xva, y_train.iloc[va])])
        oof[va] = m.predict_proba(Xva)[:, 1]
        fold_auc.append(roc_auc_score(y_train.iloc[va], oof[va]))
    return roc_auc_score(y_train, oof), np.array(fold_auc)

# --- полные (train+test) данные для групповых статистик ---
KEY = ['online_boarding','inflight_wifi_service','inflight_entertainment','class','type_of_travel']
ZERO = ['inflight_wifi_service','online_boarding','ease_of_online_booking']
cd = coded.copy()
cd['orig_prob']  = np.r_[X_prof_orig['orig_prob'].values, np.asarray(p_te)]
p = np.clip(cd['orig_prob'].values, 1e-6, 1-1e-6); cd['orig_logit'] = np.log(p/(1-p))
g = cd.groupby('flight_distance')

def tr_part(df): return df.iloc[:n].reset_index(drop=True)

blocks = {}
# B1: разброс внутри маршрута (std) по ключевым признакам и orig_logit
blocks['B1_std'] = tr_part(pd.DataFrame({f'fds_{c}': g[c].transform('std').astype('float32')
                                         for c in KEY + ['orig_logit']}))
# B2: orig-сигнал на уровне маршрута
blocks['B2_orig_route'] = tr_part(pd.DataFrame({
    'fdm_orig_prob':  g['orig_prob'].transform('mean').astype('float32'),
    'fdm_orig_logit': g['orig_logit'].transform('mean').astype('float32')}))
# B3: отклонение строки от профиля её маршрута (взаимодействие строка x маршрут)
dev = {f'dev_{c}': (cd[c] - g[c].transform('mean')).astype('float32') for c in KEY}
dev['dev_orig_logit'] = (cd['orig_logit'] - g['orig_logit'].transform('mean')).astype('float32')
blocks['B3_dev'] = tr_part(pd.DataFrame(dev))
# B4: доля нулей ("не пользовался сервисом") на маршруте
blocks['B4_zero_share'] = tr_part(pd.DataFrame({f'fdz_{c}': (cd[c]==0).groupby(cd['flight_distance']).transform('mean').astype('float32')
                                                for c in ZERO}))
# B5: профиль в разрезе (маршрут x type_of_travel)
g2 = cd.groupby(['flight_distance','type_of_travel'])
blocks['B5_seg_profile'] = tr_part(pd.DataFrame({f'fdt_{c}': g2[c].transform('mean').astype('float32')
                                                 for c in ['online_boarding','inflight_wifi_service','inflight_entertainment','class','orig_logit']}))

# --- запуск ---
ref_auc, ref_f = run_cv(X_prof_orig)
print(f'REF (prof+orig, baseline params): {ref_auc:.5f}')
results = {}
for name, blk in blocks.items():
    auc, f = run_cv(pd.concat([X_prof_orig.reset_index(drop=True), blk], axis=1))
    results[name] = auc
    print(f'{name:16s} {auc:.5f} | Δ {auc-ref_auc:+.5f} | folds', np.round(f-ref_f, 5))

good = [k for k, v in results.items() if v - ref_auc > 0.0001]
print('кандидаты для объединения:', good)

REF (prof+orig, baseline params): 0.96048
B1_std           0.96046 | Δ -0.00002 | folds [ 5.0e-05  3.0e-05 -4.0e-05 -2.0e-05 -1.7e-04]
B2_orig_route    0.96048 | Δ -0.00000 | folds [ 4.0e-05  8.0e-05 -1.2e-04  1.1e-04 -1.5e-04]
B3_dev           0.96046 | Δ -0.00002 | folds [-8.0e-05  2.0e-05 -2.0e-05  1.6e-04 -2.4e-04]
B4_zero_share    0.96051 | Δ +0.00003 | folds [ 5.e-05  6.e-05 -2.e-05 -3.e-05  6.e-05]
B5_seg_profile   0.96047 | Δ -0.00001 | folds [ 9.0e-05 -8.0e-05  2.0e-05 -1.0e-05 -1.1e-04]
кандидаты для объединения: []


In [20]:
import numpy as np, pandas as pd

p = np.clip(np.asarray(p_te), 1e-6, 1-1e-6)
X_te_final = pd.concat([X_test.reset_index(drop=True),
                        PROF.iloc[n:].reset_index(drop=True)], axis=1)
X_te_final['orig_prob']  = np.asarray(p_te)
X_te_final['orig_logit'] = np.log(p/(1-p))

X_tr_final = X_prof_orig.reset_index(drop=True)
X_te_final = X_te_final[X_tr_final.columns]            # тот же порядок колонок
assert list(X_te_final.columns) == list(X_tr_final.columns)
print(X_tr_final.shape, X_te_final.shape)              # ожидаем (699635, ~45), (299844, ~45)

X_tr_final.to_parquet('/kaggle/working/X_final_train.parquet')
X_te_final.to_parquet('/kaggle/working/X_final_test.parquet')
y_train.reset_index(drop=True).to_frame('y').to_parquet('/kaggle/working/y_train.parquet')

(699635, 45) (299844, 45)
